# Notebook #1: LSTM Autoencoder Anomaly Detection

## 1.0. Set Up the Environment and Import Libraries

In [1]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

# Import scikit-learn ML preprocessing, regression, and evaluation metrics
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    recall_score, precision_score, average_precision_score, precision_recall_curve,
    r2_score, fbeta_score, roc_auc_score, roc_curve
)

# Import PyTorch libraries and neural network building blocks
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Import the copy library for duplicating model states or objects
import copy

# Import Plotly data visualization libraries
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Initialize the hardware device (use GPU if available, otherwise CPU) and print the status
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


## 2.0. Load in and Inspect the Data From the Parquet File

In [3]:
# Store the filepath to the data parquet file
filepath = "../data/ds08a_rows.parquet"

# Read in and store the parquet file containing the dataset
df = pd.read_parquet(filepath).drop(columns=['split'])

# Display the loaded dataset's shape, column names, and first few rows
print(f"Dataset Shape: {df.shape}\n")
print(f"Dataset Columns: {df.columns}\n")
display(df.head())

Dataset Shape: (67328, 49)

Dataset Columns: Index(['raw_idx', 'alt', 'Mach', 'TRA', 'T2', 'T24', 'T30', 'T48', 'T50',
       'P15', 'P2', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'Nf', 'Nc', 'Wf',
       'T40', 'P30', 'P45', 'W21', 'W22', 'W25', 'W31', 'W32', 'W48', 'W50',
       'SmFan', 'SmLPC', 'SmHPC', 'phi', 'fan_eff_mod', 'fan_flow_mod',
       'LPC_eff_mod', 'LPC_flow_mod', 'HPC_eff_mod', 'HPC_flow_mod',
       'HPT_eff_mod', 'HPT_flow_mod', 'LPT_eff_mod', 'LPT_flow_mod', 'RUL',
       'unit', 'cycle', 'Fc', 'hs', 'EPR'],
      dtype='str')



,raw_idx,alt,Mach,TRA,T2,T24,T30,T48,T50,P15,...,HPT_eff_mod,HPT_flow_mod,LPT_eff_mod,LPT_flow_mod,RUL,unit,cycle,Fc,hs,EPR
0,0,3003.0,0.392490,84.286507,523.610962,632.830627,1525.667236,1953.949585,1319.505371,20.431551,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.144806
1,76,3757.0,0.414981,84.286507,522.673950,631.582214,1521.578369,1947.109619,1312.881836,20.102882,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.134743
2,152,4676.0,0.417879,76.024849,519.527405,620.124512,1482.610229,1877.180664,1275.688599,18.987900,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.103322
3,228,5523.0,0.430164,77.079529,517.440247,618.549683,1480.296631,1875.717041,1271.735474,18.577042,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.101272
4,304,6591.0,0.427392,77.958427,513.253784,614.360046,1473.019409,1868.009644,1264.644653,17.865288,...,-0.000612,-0.000411,-0.001029,-0.000784,71.0,1,1,1,1,1.105557


In [4]:
# Display counts of healthy vs. degraded time steps (broken down by 'Fc' and engine unit)
print(f"Total number of healthy time steps: {len(df[df['hs'] == 1.0])}")
print(f"Total number of degraded/unhealthy time steps: {len(df[df['hs'] == 0.0])}")

print("\nHealthy and Unhealthy Fc/unit time steps:")
display(df.groupby(['Fc', 'unit', 'hs']).size().reset_index(name='timesteps'))

Total number of healthy time steps: 21658
Total number of degraded/unhealthy time steps: 45670

Healthy and Unhealthy Fc/unit time steps:


,Fc,unit,hs,timesteps
0,1,1,0,2469
1,1,1,1,1757
2,1,3,0,2799
3,1,3,1,1681
4,1,9,0,3619
5,1,9,1,1644
6,1,10,0,2679
7,1,10,1,1754
8,1,15,0,2541
9,1,15,1,1474


## 3.0. Preprocess the Data

### 3.1. Split the Data into Train/Validation/Test Splits

Before we preprocess the data, we will begin by splitting the entire dataset into train, validation, and test splits. We hardcode which engines will be used for each split since we want to use the same ones as our Isolation Forest model for easier comparison of model performance. The **15 total engines are divided into 9 training, 3 validation, and 3 test engines**, stratified by flight class so that `'Fc' = 1`, `'Fc' = 2`, and `'Fc' = 3` appear in every partition. We also verify that the number of rows for each of the data splits matches with what was used in the Isolation Forest model notebooks.

In [5]:
# Store the fixed engine units to be used for creating the train, val, and test data splits
TRAIN_UNITS = [3, 9, 15, 5, 7, 8, 2, 6, 13]
VAL_UNITS = [10, 4, 14]
TEST_UNITS = [1, 11, 12]

# Filter for and store train, val, test data splits using the fixed engine units
train = df[df['unit'].isin(TRAIN_UNITS)].copy()
val = df[df['unit'].isin(VAL_UNITS)].copy()
test = df[df['unit'].isin(TEST_UNITS)].copy()

# Verify the train, val, test split 'Fc'/'unit' time step counts
print(f"Training samples: {len(train)}")
print("Training set 'Fc'/'unit' time steps:")
display(train.groupby(['Fc', 'unit']).size().reset_index(name='rows'))

print(f"\nValidation samples: {len(val)}")
print("Validation set 'Fc'/'unit' time steps:")
display(val.groupby(['Fc', 'unit']).size().reset_index(name='rows'))

print(f"\nTest samples: {len(test)}")
print("Test set 'Fc'/'unit' time steps:")
display(test.groupby(['Fc', 'unit']).size().reset_index(name='rows'))

Training samples: 40173
Training set 'Fc'/'unit' time steps:


,Fc,unit,rows
0,1,3,4480
1,1,9,5263
2,1,15,4015
3,2,5,3864
4,2,7,4684
5,2,8,4521
6,3,2,4108
7,3,6,4500
8,3,13,4738



Validation samples: 13898
Validation set 'Fc'/'unit' time steps:


,Fc,unit,rows
0,1,10,4433
1,2,4,4844
2,3,14,4621



Test samples: 13257
Test set 'Fc'/'unit' time steps:


,Fc,unit,rows
0,1,1,4226
1,2,11,4316
2,3,12,4715


### 3.3. Filter the Training Split to Contain Only Healthy Data

We must ensure that our **training dataset only contains data from when the engines are perfectly healthy** (`'hs' == 1`). This is because our LSTM autoencoder can only be trained on healthy data so that it struggles to reconstruct data that is considered degraded/unhealthy (`'hs' == 0`).

In [6]:
# Filter training data to retain only healthy operational states (hs == 1.0)
train_healthy = train[train['hs'] == 1].copy()

# Display the number of healthy and unhealthy training rows
print(f"Number of Total Training Rows: {len(train)}")
print(f"Number of Healthy Training Rows: {len(train_healthy)}")
print(f"Number of Unhealthy Training Rows: {len(train) - len(train_healthy)}")
print(f"Percentage of Healthy Training Rows: {len(train_healthy) / len(train) * 100}%")

Number of Total Training Rows: 40173
Number of Healthy Training Rows: 12518
Number of Unhealthy Training Rows: 27655
Percentage of Healthy Training Rows: 31.16023199661464%


In [7]:
# Display the number of healthy and unhealthy validation rows
val_healthy = val[val['hs'] == 1]
print(f"Number of Total Training Rows: {len(val)}")
print(f"Number of Healthy Training Rows: {len(val_healthy)}")
print(f"Number of Unhealthy Training Rows: {len(val) - len(val_healthy)}")
print(f"Percentage of Healthy Training Rows: {len(val_healthy) / len(val) * 100}%")

# Display the number of healthy and unhealthy test rows
test_healthy = test[test['hs'] == 1]
print(f"\nNumber of Total Training Rows: {len(test)}")
print(f"Number of Healthy Training Rows: {len(test_healthy)}")
print(f"Number of Unhealthy Training Rows: {len(test) - len(test_healthy)}")
print(f"Percentage of Healthy Training Rows: {len(test_healthy) / len(test) * 100}%")

Number of Total Training Rows: 13898
Number of Healthy Training Rows: 4539
Number of Unhealthy Training Rows: 9359
Percentage of Healthy Training Rows: 32.65937544970499%

Number of Total Training Rows: 13257
Number of Healthy Training Rows: 4601
Number of Unhealthy Training Rows: 8656
Percentage of Healthy Training Rows: 34.70619295466546%


From the outputs here, we can see that for each dataset (training, validation, and test), there are significantly more rows in which engines were labeled to be degraded/unhealthy (`'hs' == 0`). This makes sense as degradation begins earlier on in the NCMAPSS data. This **target variable data imbalance** will have implications for our test results and interpretations, particularly for our PR-related scores curves.

### 3.4. Implement Operating-Condition Regression Normalization

Before further preprocessing the data or feeding it into the LSTM autoencoder, we conduct operating-condition regression normalization on the sensor data (with the help of an MLP regression model)  in order to isolate true engine degradation from fluctuations in the operating conditions.

In [8]:
# Create a list to store the raw sensor, residual sensor, and operating column names
sensor_columns = ['T24', 'T30', 'T48', 'T50', 'P15', 'P2', 'P24', 'Ps30', 'P50', 'Nf', 'Nc', 'Wf']
operating_columns = ['alt', 'Mach', 'TRA', 'T2']
RESIDUAL_COLUMNS = [f"{col}_res" for col in sensor_columns]

In [9]:
# Standardize the operating conditions across train, validation, and test sets
operating_scaler = StandardScaler()
train_operating_scaled = operating_scaler.fit_transform(train_healthy[operating_columns])
val_operating_scaled = operating_scaler.transform(val[operating_columns])
test_operating_scaled = operating_scaler.transform(test[operating_columns])

# Standardize the healthy training sensor values to ensure stable MLP regressor training convergence
sensor_scaler = StandardScaler()
train_sensors_scaled = sensor_scaler.fit_transform(train_healthy[sensor_columns])

# Train the MLP regressor to learn how normal sensors behave from operating conditions
mlp_reg = MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=200, random_state=42)
mlp_reg.fit(train_operating_scaled, train_sensors_scaled)

# Predict sensor values for train, val, and test sets and inverse-transform them back to original units
train_mlp_preds = sensor_scaler.inverse_transform(mlp_reg.predict(train_operating_scaled))
val_mlp_preds = sensor_scaler.inverse_transform(mlp_reg.predict(val_operating_scaled))
test_mlp_preds = sensor_scaler.inverse_transform(mlp_reg.predict(test_operating_scaled))

# Calculate and store the residuals (Actual - Predicted) for the train, val, and test sets
train_healthy[RESIDUAL_COLUMNS] = train_healthy[sensor_columns].values - train_mlp_preds
val[RESIDUAL_COLUMNS] = val[sensor_columns].values - val_mlp_preds
test[RESIDUAL_COLUMNS] = test[sensor_columns].values - test_mlp_preds

# Display the MLP regression validation R² score
print(f"MLPRegressor Validation R² Score: {r2_score(val[sensor_columns], val_mlp_preds)}")

MLPRegressor Validation R² Score: 0.9970864653587341


References:

* Sánchez, P., Reyes, K., Radu, B., & Fernández, E. (2026). *Early fault detection on CMAPSS with unsupervised LSTM autoencoders*. arXiv. https://doi.org/10.48550/arXiv.2601.10269

### 3.5. Scale the Sensor Residual Columns for the Neural Network

Since we calculated the our residuals in the previous step using the original units of the sensors, it is the case that the values of these columns can cover vastly different numerical ranges. As such, we use **`StandardScaler`** to normalize variance across all sensor features so they are all treated equally.

In [10]:
# Initialize a StandardScaler to normalize sensor residual feature variance
residual_scaler = StandardScaler()

# Fit the scaler on healthy training sensor residuals and transform train, validation, and test sensor residuals
train_healthy[RESIDUAL_COLUMNS] = residual_scaler.fit_transform(train_healthy[RESIDUAL_COLUMNS])
val[RESIDUAL_COLUMNS] = residual_scaler.transform(val[RESIDUAL_COLUMNS])
test[RESIDUAL_COLUMNS] = residual_scaler.transform(test[RESIDUAL_COLUMNS])

References:

* Timoficiuc, M. (2023, June 14). Predicting jet engine failures with NASA’s C-MAPSS dataset and LSTM: A practical guide. Medium. https://medium.com/@mihaitimoficiuc/predicting-jet-engine-failures-with-nasas-c-mapss-dataset-and-lstm-a-practical-guide-to-85b9513ea9ed

### 3.7. Define a Helper Function for Creating Residual Sliding Windows

For our LSTM autoencoder, we will have to transform our raw, sequential sensor data into **overlapping sliding windows**. We do this so that our model can capture temporal patterns, allowing it to learn to how sensor behavior and degradation evolves over time.

In [11]:
def create_residual_sliding_windows(df, residual_cols, window_size):
  """
  Creates overlapping per-engine sliding windows of time-series residual data
  to be used in LSTM autoencoder for temporal anomaly detection
  """
  residual_windows = []
  health_status_windows = []
  unit_ids = []
  flight_conditions = []
  window_end_steps = []
  window_ranges = []

  # Iterate through each individual engine in the provided data
  for unit_id in df['unit'].unique():
    # Filter the data to only include the respective engine dat
    engine_data = df[df['unit'] == unit_id]

    # Extract and store the 'Fc' value related to the engine
    flight_condition = int(engine_data['Fc'].iloc[0])

    # Extract and store the sensor residuals (features) and health status labels (target)
    residual_values = engine_data[residual_cols].values
    health_status_values = engine_data['hs'].values

    # Only process engines that have enough time steps to make up one window sequence
    if len(residual_values) >= window_size:
      # Iterate through each sliding window step for the engine
      for i in range(len(residual_values) - window_size + 1):
        # Extract and store the sensor residuals for the current window sequence
        residual_windows.append(residual_values[i : i + window_size])

        # Extract and store the health status at the final step of the current window
        health_status_windows.append(health_status_values[i + window_size - 1])

        # Store additional data corresponding to this specific window
        unit_ids.append(unit_id)
        flight_conditions.append(flight_condition)
        window_end_steps.append(i + window_size)
        window_ranges.append(f"{i + 1}-{i + window_size}")

  # Return a packaged dictionary of the PyTorch tensors and additional data
  return {
    'tensor_residuals': torch.tensor(np.array(residual_windows), dtype=torch.float32),
    'tensor_health_statuses': torch.tensor(np.array(health_status_windows), dtype=torch.float32),
    'unit_ids': np.array(unit_ids),
    'flight_conditions': np.array(flight_conditions),
    'window_end_steps': np.array(window_end_steps),
    'window_ranges': np.array(window_ranges)
  }

References:

* Brownlee, J. (2018, August 21). *Introduction to time-series forecasting with long short-term memory networks in Python*. Machine Learning Mastery. https://machinelearningmastery.com/lstm-for-time-series-prediction-in-pytorch/

* Sánchez, P., Reyes, K., Radu, B., & Fernández, E. (2026). *Early fault detection on CMAPSS with unsupervised LSTM autoencoders*. arXiv. https://doi.org/10.48550/arXiv.2601.10269

* Timoficiuc, M. (2023, June 14). *Predicting jet engine failures with NASA’s C-MAPSS dataset and LSTM: A practical guide*. Medium. https://medium.com/@mihaitimoficiuc/predicting-jet-engine-failures-with-nasas-c-mapss-dataset-and-lstm-a-practical-guide-to-85b9513ea9ed

## 4.0. Define the LSTM Autoencoder Structure

We wish to model temporal degradation patterns across multi-sensor engine data by using a funnel **Encoder-Decoder LSTM Autoencoder**. We reference the model architecture outlined by Sanchez et al. (2026) for time-series anomaly detection.

In our network, the encoder segment uses three stacked LSTM layers to funnel and compress temporal sliding windows into a fixed bottleneck latent representation. Our decoder segment follows a mirrored approach utilizing three stacked LSTM layers and a final linear projection layer to expand and reconstruct the bottleneck latent vector back to the original sensor residual sequence. For both segments, **Dropout** layers are incorporated between layers to prevent overfitting on the training data.

In [12]:
class LSTMEncoder(nn.Module):
  def __init__(self, sensor_dim, hidden_size, dropout):
    super().__init__()
    # Determine the sizes of the intermediate layer and bottleneck latent representation
    middle_dim = int(hidden_size // 2)
    latent_dim = int(hidden_size // 4)

    # 3-layer LSTM funnel compression with dropout: sensor_dim -> hidden_size -> middle_dim -> latent_dim
    # LSTM layer #1: Project the input sliding window sensor dimensions to the passed hidden_size dimension
    self.lstm1 = nn.LSTM(input_size=sensor_dim, hidden_size=hidden_size, num_layers=1, batch_first=True)
    self.dropout1 = nn.Dropout(dropout)

    # LSTM layer #2: Compress the passed hidden_size dimension down to the intermediate layer dimension
    self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=middle_dim, num_layers=1, batch_first=True)
    self.dropout2 = nn.Dropout(dropout)

    # LSTM layer #3: Compress the intermediate layer dimension down to the final bottleneck latent representation
    self.lstm3 = nn.LSTM(input_size=middle_dim, hidden_size=latent_dim, num_layers=1, batch_first=True)

  def forward(self, sensor_residuals):
    # Input shape: (batch_size, seq_len, sensor_dim)

    # Pass through the encoder's first LSTM layer and implement dropout
    # Shape: --> (batch_size, seq_len, hidden_size)
    output, _ = self.lstm1(sensor_residuals)
    output = self.dropout1(output)

    # Pass through the encoder's second LSTM layer and implement dropout
    # Shape: --> (batch_size, seq_len, middle_dim)
    output, _ = self.lstm2(output)
    output = self.dropout2(output)

    # Pass through the encoder's final LSTM layer and extract hidden state representation
    # Shape: --> (num_layers, batch_size, latent_dim) --> (batch_size, latent_dim)
    _, (hidden_n, _) = self.lstm3(output)
    latent = hidden_n[-1]
    return latent

class LSTMDecoder(nn.Module):
  def __init__(self, sensor_dim, hidden_size, dropout):
    super().__init__()
    # Determine the sizes of the intermediate layer and bottleneck latent representation
    middle_dim = int(hidden_size // 2)
    latent_dim = int(hidden_size // 4)

    # 3-layer LSTM funnel expansion with dropout: latent_dim -> middle_dim -> hidden_size -> sensor_dim
    # LSTM layer #1: Expand bottleneck latent representation dimension back up to intermediate layer dimension
    self.lstm1 = nn.LSTM(input_size=latent_dim, hidden_size=middle_dim, num_layers=1, batch_first=True)
    self.dropout1 = nn.Dropout(dropout)

    # LSTM layer #2: Expand intermediate layer dimension back up to the passed 'hidden_size' dimension
    self.lstm2 = nn.LSTM(input_size=middle_dim, hidden_size=hidden_size, num_layers=1, batch_first=True)
    self.dropout2 = nn.Dropout(dropout)

    # LSTM layer #3: Maintain the expanded 'hidden_size' sequence representation across temporal steps
    self.lstm3 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=1, batch_first=True)

    # Final linear layer to project passed 'hidden_size' dimension back to the original sensor dimensions
    self.output_layer = nn.Linear(hidden_size, sensor_dim)

  def forward(self, latent, sensor_residuals):
    # Input latent shape: (batch_size, latent_dim)
    # Input sensor_residuals shape: (batch_size, seq_len, sensor_dim)

    # Add a time dimension to the bottleneck latent vector to create a temporal context vector
    # Shape: --> (batch_size, 1, latent_dim)
    context_vector = latent.unsqueeze(1)

    # Broadcast and expand the context vector across the entire sliding window sequence length
    # Shape: --> (batch_size, seq_len, latent_dim)
    expanded_context = context_vector.expand(-1, sensor_residuals.size(1), -1)

    # Pass through the decoder's first LSTM layer and implement dropout
    # Shape: --> (batch_size, seq_len, middle_dim)
    output, _ = self.lstm1(expanded_context)
    output = self.dropout1(output)

    # Pass through the decoder's second LSTM layer and implement dropout
    # Shape: --> (batch_size, seq_len, hidden_size)
    output, _ = self.lstm2(output)
    output = self.dropout2(output)

    # Pass through the decoder's final LSTM layer
    # Shape: --> (batch_size, seq_len, hidden_size)
    output, _ = self.lstm3(output)

    # Project dimension back to original sensor dimensions via linear layer
    # Shape: --> (batch_size, seq_len, sensor_dim)
    return self.output_layer(output)

class LSTMAutoencoder(nn.Module):
  def __init__(self, sensor_dim, hidden_size, dropout):
    super().__init__()
    # Initialize the funnel compression encoder and symmetric expansion decoder blocks
    self.encoder = LSTMEncoder(sensor_dim, int(hidden_size), dropout)
    self.decoder = LSTMDecoder(sensor_dim, int(hidden_size), dropout)

  def forward(self, sensor_residuals):
    # Input shape: (batch_size, seq_len, sensor_dim)

    # Compress sensor residual data into bottleneck latent vector via the encoder
    # Shape: --> (batch_size, latent_dim)
    latent = self.encoder(sensor_residuals)

    # Reconstruct the residual sensor data via the decoder
    # Shape: --> (batch_size, seq_len, sensor_dim)
    reconstructed_residuals = self.decoder(latent, sensor_residuals)

    return reconstructed_residuals

References:


* Curiousily. (2020, March 22). *Time series anomaly detection using LSTM autoencoders with PyTorch in Python*. https://curiousily.com/posts/time-series-anomaly-detection-using-lstm-autoencoder-with-pytorch-in-python/

* GeeksforGeeks. (2026a, February 9). *How to use PyTorch for anomaly detection?* https://www.geeksforgeeks.org/deep-learning/how-to-use-pytorch-for-anomaly-detection/

* GeeksforGeeks. (2025b, August 5). *Implementing an autoencoder in PyTorch*. https://www.geeksforgeeks.org/deep-learning/implementing-an-autoencoder-in-pytorch/

* GeeksforGeeks. (2025c, October 9).* Long short term memory (LSTM) networks using PyTorch*. https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/

* Lukas. (n.d.). *Time series anomaly detection with autoencoders* [Jupyter notebook]. Kaggle. https://www.kaggle.com/code/lkskrk/time-series-anomaly-detection-with-autoencoders

* Sánchez, P., Reyes, K., Radu, B., & Fernández, E. (2026, January 15). *Early fault detection on CMAPSS with unsupervised LSTM autoencoders*. arXiv. https://doi.org/10.48550/arXiv.2601.10269

* Timoficiuc, M. (2023, June 14). *Predicting jet engine failures with NASA’s C-MAPSS dataset and LSTM: A practical guide*. Medium. https://medium.com/@mihaitimoficiuc/predicting-jet-engine-failures-with-nasas-c-mapss-dataset-and-lstm-a-practical-guide-to-85b9513ea9ed

## 5.0. Train the LSTM Autoencoder and Conduct Hyperparameter Tuning using the Validation Set

### 5.1. Create a Helper Function for Training the LSTM Autoencoder Using Early Stopping

We will create the `train_lstm_autoencoder()` function that manages the LSTM autoencoder training process and tracks validation loss across epochs for our hyperparameter tuning later on. As a default, we will set the maximum number of epochs equal to 200 (`'epochs' = 200`) and will use a a patience of 5 epochs (`'patience' = 5)` for early stopping to prevent overfitting and save computation time. A patience of 5 means that training will stop once the validation loss fails to improve for 5 consecutive epochs. For this early stopping evaluation, the validation set is restricted exclusively to healthy engine states in order to ensure that the model optimizes its generalization capability on normal healthy dynamics.

In [13]:
def train_lstm_autoencoder(model, train_loader, val_loader, optimizer, loss_function, epochs=200, patience=5):
  """
  Trains the LSTM autoencoder model using early stopping
  """
  # Create variables to store the average train/validation losses over the epochs
  train_losses = []
  val_losses = []

  # Initialize best validation loss tracking, model parameters copy, and early stopping patience counter
  best_val_loss = float('inf')
  best_parameters = copy.deepcopy(model.state_dict())
  patience_counter = 0

  for epoch in range(epochs):
    # -- TRAINNG PHASE --
    # Set the model to training mode and monitor the epoch's running training loss
    model.train()
    running_train_loss = 0.0

    # Iterate through the training data loader batches
    for sensor_batch in train_loader:
      # Extract and move the sensor tensor to the hardware device (CPU or GPU)
      sensor_batch = sensor_batch[0].to(DEVICE)

      # Reset gradients from the previous step
      optimizer.zero_grad()

      # Forward pass: Retrieve autoencoder reconstructed sliding window sensor residuals
      reconstructions = model(sensor_batch)

      # Calculate the reconstruction loss for the batch
      loss = loss_function(reconstructions, sensor_batch)

      # Backward pass: Compute gradients via backpropagation
      loss.backward()

      # Update model parameters with a single optimization step
      optimizer.step()

      # Accumulate total weighted training loss for the epoch
      running_train_loss += loss.item() * sensor_batch.size(0)

    # -- VALIDATION PHASE --
    # Set the model to evaluation mode and monitor the epoch's running validaion loss
    model.eval()
    running_val_loss = 0.0

    # Disable gradient computation for efficiency
    with torch.no_grad():
      # Iterate through the validation data loader batches
      for sensor_batch in val_loader:
        # Extract and move the sensor tensor to the hardware device (CPU or GPU)
        sensor_batch = sensor_batch[0].to(DEVICE)

        # Forward pass: Retrieve autoencoder reconstructed sliding window sensor residuals
        reconstructions = model(sensor_batch)

        # Calculate the reconstruction loss for the batch
        loss = loss_function(reconstructions, sensor_batch)

        # Accumulate total weighted validation loss for the epoch
        running_val_loss += loss.item() * sensor_batch.size(0)

    # Calculate overall weighted average train/validation losses for the epoch
    epoch_avg_train_loss = running_train_loss / len(train_loader.dataset)
    epoch_avg_val_loss = running_val_loss / len(val_loader.dataset)

    # Store epoch losses in their respective tracking lists
    train_losses.append(epoch_avg_train_loss)
    val_losses.append(epoch_avg_val_loss)

    # -- EARLY STOPPING --
    # Check if the current epoch has achieved a new lowest validation loss
    if epoch_avg_val_loss < best_val_loss:
      # Update the lowest validation loss to current one
      best_val_loss = epoch_avg_val_loss

      # Save a deep copy of the current best model parameters
      best_parameters = copy.deepcopy(model.state_dict())

      # Reset the patience counter after validation improvement
      patience_counter = 0
    else:
      # Increment patience counter if validation loss failed to improve
      patience_counter += 1

      # Stop training early if the patience threshold is met
      if patience_counter >= patience:
        break

  # Restore the saved model weights corresponding to the lowest validation loss
  model.load_state_dict(best_parameters)

  # Return the optimized model, train/validation losses, and the lowest validation loss
  return model, train_losses, val_losses, best_val_loss

References:
* Ghimirey, B. (2023). *Running loss vs. average loss: The beginner’s guide to training loss in PyTorc*h. Medium. https://medium.com/@bikashghimirey797/running-loss-vs-average-loss-the-beginners-guide-to-training-loss-in-pytorch-d35bbf1d9ef0

* One-Off Coder. (2026, April 10). *6. Early stopping*. PyTorch, No Tears. https://learn-pytorch.oneoffcoder.com/early-stopping.html

### 5.2. Create a Helper Function for Model Evaluation on the Validation/Test Sets

We will create the `evaluate_lstm_autoencoder()` function to be used for calculating anomaly scores and evaluating the performance of our validation and test sets later on. We outline key steps involved in this evaluation function below:

* **Summing MAE across sensors**: We define our reconstruction error as the sum of Mean Absolute Errors (MAEs) across all individual sensor dimensions at each sliding window time step to avoid relying on any single sensor for reliable anomaly detection.
*  **Rolling smoothing of anomaly scores (Optional)**: We include an optional rolling median filter to smooth the anomaly scores across consecutive time windows in order to reduce high-frequency noise.
* **Inverting the target health status labels**: We invert the ground-truth health state ('hs') labels so that anomalies (degraded engines) are treated as the positive class (`1`) for evaluation metrics like ROC-AUC and PR-AUC. Because the original dataset designated healthy engines as `1.0` and degraded engines as `0.0`, this inversion aligns the labels with standard binary classification conventions.

In [14]:
def evaluate_lstm_autoencoder(model, loader, data_dict, smooth=True, smooth_window=5):
  """
  Evaluates the LSTM autoencoder on the validation or test dataset loader by
  calculating the anomaly scores and ROC-AUC and PR-AUC performance metrics
  """
  model.eval()
  scores = []

  # Disable gradient computation for efficiency
  with torch.no_grad():
    # Iterate through the validation/test data loader batches
    for sensor_batch in loader:
      # Extract and move the sensor tensor to the hardware device (CPU or GPU)
      sensor_batch = sensor_batch[0].to(DEVICE)

      # Forward pass: Retrieve autoencoder reconstructed sliding window sensor residuals
      reconstructions = model(sensor_batch)

      # Calculate absolute reconstruction errors element-wise across the batch
      abs_errors = torch.abs(reconstructions - sensor_batch)

      # Calculate the mean absolute error (MAE) across the sliding window time dimension (dim=1)
      time_averaged_sensor_errors = torch.mean(abs_errors, dim=1)

      # Sum the MAE across all sensor features (dim=1) to get the batch window error scores
      window_anomaly_scores = time_averaged_sensor_errors.sum(dim=1)

      # Move the batch scores from GPU/hardware device to CPU and extend the scores list
      scores.extend(window_anomaly_scores.cpu().numpy())

  # Convert the accumulated score list into a NumPy array
  anomaly_scores = np.array(scores)

  # Optionally apply rolling median smoothing to reduce high-frequency noise in anomaly scores
  if smooth:
    anomaly_scores = pd.Series(anomaly_scores).rolling(smooth_window, min_periods=1, center=True).median().to_numpy()

  # Invert the health status labels so that degraded/unhealthy engines represent the positive anomaly class
  # ('hs' = 1 = Healthy -> 0 = Normal, 'hs' = 0 = Degraded -> 1 = Anomaly)
  anomaly_labels = 1.0 - data_dict['tensor_health_statuses'].numpy()

  # Calculate classification performance metrics (ROC-AUC and PR-AUC)
  roc_auc = roc_auc_score(anomaly_labels, anomaly_scores)
  pr_auc = average_precision_score(anomaly_labels, anomaly_scores)

  # Return the performance metrics, anomaly scores, and inverted health status labels
  return roc_auc, pr_auc, anomaly_scores, anomaly_labels

* GeeksforGeeks. (2026, February 9). How to use PyTorch for anomaly detection? https://www.geeksforgeeks.org/deep-learning/how-to-use-pytorch-for-anomaly-detection/

### 5.3. Create a Helper Function for Summarizing Segmented ROC-AUC/PR-AUC Metrics

We will create the `create_segmented_metric_summary()` to create a DataFrame that we can use to evaluate model performance across distinct flight condition regimes. To determine whether our LSTM autoencoder model generalizes effectively to unseen data across different flight condition regimes, we include both the **overall ROC-AUC and PR-AUC scores** for the validation/test set as well as **stratified metrics for each individual flight condition (`Fc`)**. These results are compiled into a formatted summary DataFrame for comparative analysis.

In [15]:
def create_segmented_metric_summary(df, anomaly_labels, anomaly_scores, overall_roc_auc, overall_pr_auc, group_col='Fc'):
  """Creates a DataFrame of overall and segment-specific (e.g., flight conditions) ROC-AUC and PR-AUC scores"""
  # Initialize dictionaries to store overall ROC-AUC and PR-AUC metrics
  roc_auc_dict = {'Overall': round(overall_roc_auc, 4)}
  pr_auc_dict = {'Overall': round(overall_pr_auc, 4)}

  # Iterate through each distinct group (e.g., flight condition) to calculate segmented performance and add to dictionaries
  for group_val, group_data in df.groupby(group_col):
    roc_auc_dict[f'Fc = {group_val}'] = roc_auc_score(
      group_data['Health Status (Numeric)'],
      group_data['Residual Reconstruction Error (Cumulative MAE)']
    )
    pr_auc_dict[f'Fc = {group_val}'] = average_precision_score(
      group_data['Health Status (Numeric)'],
      group_data['Residual Reconstruction Error (Cumulative MAE)']
    )

  # Combine scores into a single DataFrame, transpose for readability, and assign descriptive column names
  summary_df = pd.DataFrame([roc_auc_dict, pr_auc_dict], index=['ROC-AUC', 'PR-AUC'])
  return summary_df.transpose()

### 5.4. Hyperparamter Tuning using Simple Manual Grid Search

We chose to use a fixed sliding window sequence length of 10 time steps/rows because we felt that it was sufficient enough to capture immediate sensor changes and short-term trends. In choosing this window size, we also considered how we had downsampled the original raw NCMAPSS data at different rates depending on the engine flight condition (`'Fc'`):

* `'Fc' == 1` (1 sample/76 seconds): `'window_size' = 10` will capture **12.7 minutes** of engine operation.
* `'Fc' == 2` (1 sample/127 seconds): `'window_size' = 10` will capture **21.2 minutes** of engine operation.
* `'Fc' == 3` (1 sample/180 seconds): `'window_size' = 10` will capture **30 minutes** of engine operation.

By factoring in these downsampling rates, we believe that a 10-step sliding window sequence provides a reliable temporal background across varying flight regimes.

We chose a fixed standard batch size of 64 because it represents a robust standard in time-series deep learning models. Also, it has worked well for us in regards to model optimization stability and processing speed.

In [16]:
# Store fixed values to use for sliding window sequence length and batch size
WINDOW_SIZE = 10
BATCH_SIZE = 64

# Create the residual sliding window tensors for the train and validation sets
val_healthy = val[val['hs'] == 1.0].reset_index(drop=True)
train_data = create_residual_sliding_windows(train_healthy, RESIDUAL_COLUMNS, window_size=WINDOW_SIZE)
val_healthy_data = create_residual_sliding_windows(val_healthy, RESIDUAL_COLUMNS, window_size=WINDOW_SIZE)
val_full_data = create_residual_sliding_windows(val, RESIDUAL_COLUMNS, window_size=WINDOW_SIZE)

# Wrap the train/validation tensors in PyTorch datasets and create the train/validation DataLoaders
train_dataset = TensorDataset(train_data['tensor_residuals'])
val_healthy_dataset = TensorDataset(val_healthy_data['tensor_residuals'])
val_full_dataset = TensorDataset(val_full_data['tensor_residuals'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_healthy_loader = DataLoader(val_healthy_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_full_loader = DataLoader(val_full_dataset, batch_size=BATCH_SIZE, shuffle=False)

We create a dictionary of unique hyperparameter combinations that we will like to tune for by testing on the validation set. The hyperparameters we focus on are:

* **`'hidden_size'`** for determining the dimensionality of the LSTM encoder/decoder layers.
* **`'dropout'`** for controlling the fraction of units randomly deactivated during training.
* **`'learning_rate'`** for adjusting the optimzer step size during weight updates.

We will store the hyperparameter combinations and their validation performance scores in a pandas DataFrae that we will be able to access later on.


In [17]:
# Store the hyperparameter combinations for tuning in a list of disctionaries
param_combinations = [
  {
      'hidden_size': h,
      'dropout': d,
      'learning_rate': lr
  }
  for h in [64, 128]
  for d in [0.2, 0.3]
  for lr in [1e-3, 5e-4]
]

# Create a DataFrame to store the hyperparamter combinations and their respective results
param_combos_df = pd.DataFrame(param_combinations)

# Display the hyperparameter combinations DataFrame
print("Hyperparameter Combinations:")
display(param_combos_df)

Hyperparameter Combinations:


,hidden_size,dropout,learning_rate
0,64,0.2,0.0010
1,64,0.2,0.0005
2,64,0.3,0.0010
3,64,0.3,0.0005
4,128,0.2,0.0010
5,128,0.2,0.0005
6,128,0.3,0.0010
7,128,0.3,0.0005


Now that we have defined the hyperparameter combinations we are interested in evaluating, we can initiate our grid search tuning process. Each hyperparameter combination undergoes a single training routine capped at a **maximum of 200 epochs** (`'epochs' = 200)`, incorporating an early stopping mechanism (**with a patience of 5 epochs** or `'patience' = 5`) to halt training once the validation loss plateaus. Weh have also chosen to train models within the search space using a standard Adam optimizer and a Mean Absolute Error (`nn.L1Loss`) loss function.


Once the tuning process is complete, we will output the hyperparameter combinations and their respective validation losses and scores

In [18]:
print("=========STARTING VALIDATION SET EVALUATION!!!=========\n")

# Pre-initialize certain columns as object type to avoid list unpacking
for object_column in ['model_object', 'train_losses', 'val_losses']:
  param_combos_df[object_column] = None
  param_combos_df[object_column] = param_combos_df[object_column].astype(object)

# Iterate through each hyperparameter configuration in the search grid
for combo_number, param_combo in enumerate(param_combinations):
  # Extract and store the individual hyperparameters for the current combination
  hidden_size = param_combo['hidden_size']
  dropout = param_combo['dropout']
  learning_rate = param_combo['learning_rate']

  print(f"========== Hyperparameter Combination [{combo_number+1}/{len(param_combinations)}] ==========")
  print(f"Hidden Size: {hidden_size} | Dropout: {dropout} | Learning Rate: {learning_rate}")

  # Instantiate a new LSTM Autoencoder model instance with the hidden_size and dropout parameters
  model = LSTMAutoencoder(
    sensor_dim=len(RESIDUAL_COLUMNS),
    hidden_size=hidden_size,
    dropout=dropout
  )
  model = model.to(DEVICE)

  # Compile the model to optimize execution speed via PyTorch 2.0+ compilation
  model = torch.compile(model)

  # Train the model using early stopping and the current learning rate
  model, train_losses, val_losses, model_val_loss = train_lstm_autoencoder(
    model=model,
    train_loader=train_loader,
    val_loader=val_healthy_loader, # Use only healthy validation data for early stopping
    optimizer=optim.Adam(model.parameters(), lr=learning_rate),
    loss_function=nn.L1Loss()
  )

  # Print the best validation loss for the hyperparameter combination
  print(f"Best Validation Loss: {model_val_loss:.4f}\n")

  # Evaluate ROC-AUC and PR-AUC scores on the full validation set (healthy and unhealthy) for this combination
  val_roc_auc, val_pr_auc, _, _ = evaluate_lstm_autoencoder(model, val_full_loader, val_full_data)

  # Record validation performance, history, and trained model objects into the hyperparameter DataFrame
  param_combos_df.at[combo_number, 'best_val_loss'] = model_val_loss
  param_combos_df.at[combo_number, 'epochs'] = len(val_losses)
  param_combos_df.at[combo_number, 'val_roc_auc'] = val_roc_auc
  param_combos_df.at[combo_number, 'val_pr_auc'] = val_pr_auc
  param_combos_df.at[combo_number, 'model_object'] = model
  param_combos_df.at[combo_number, 'train_losses'] = train_losses
  param_combos_df.at[combo_number, 'val_losses'] = val_losses

print("=========FINISHING VALIDATION SET EVALUATION!!!========\n")

# Display the hyperparameter tuning results sorted by validation ROC-AUC (highest first)
print("Hyperparameter Tuning Results:")
param_combos_df = param_combos_df.sort_values(by='val_roc_auc', ascending=False)
display(param_combos_df.drop(columns=['model_object', 'train_losses', 'val_losses']))

=========STARTING VALIDATION SET EVALUATION!!!=========

========== Hyperparameter Combination [1/8] ==========
Hidden Size: 64 | Dropout: 0.2 | Learning Rate: 0.001
Best Validation Loss: 0.4113



/var/folders/x_/hnj5rbr907z8vggzlcnnt8tr0000gn/T/ipykernel_24608/1852295906.py:45: Pandas4Warning: Setting a value on a DataFrame via .at with a key that does not exist in the index is deprecated and will raise a KeyError in a future version. Use .loc instead.
  param_combos_df.at[combo_number, 'best_val_loss'] = model_val_loss
/var/folders/x_/hnj5rbr907z8vggzlcnnt8tr0000gn/T/ipykernel_24608/1852295906.py:46: Pandas4Warning: Setting a value on a DataFrame via .at with a key that does not exist in the index is deprecated and will raise a KeyError in a future version. Use .loc instead.
  param_combos_df.at[combo_number, 'epochs'] = len(val_losses)
/var/folders/x_/hnj5rbr907z8vggzlcnnt8tr0000gn/T/ipykernel_24608/1852295906.py:47: Pandas4Warning: Setting a value on a DataFrame via .at with a key that does not exist in the index is deprecated and will raise a KeyError in a future version. Use .loc instead.
  param_combos_df.at[combo_number, 'val_roc_auc'] = val_roc_auc
/var/folders/x_/hnj5r

========== Hyperparameter Combination [2/8] ==========
Hidden Size: 64 | Dropout: 0.2 | Learning Rate: 0.0005
Best Validation Loss: 0.4288

========== Hyperparameter Combination [3/8] ==========
Hidden Size: 64 | Dropout: 0.3 | Learning Rate: 0.001
Best Validation Loss: 0.4318

========== Hyperparameter Combination [4/8] ==========
Hidden Size: 64 | Dropout: 0.3 | Learning Rate: 0.0005
Best Validation Loss: 0.4349

========== Hyperparameter Combination [5/8] ==========
Hidden Size: 128 | Dropout: 0.2 | Learning Rate: 0.001
Best Validation Loss: 0.3506

========== Hyperparameter Combination [6/8] ==========
Hidden Size: 128 | Dropout: 0.2 | Learning Rate: 0.0005
Best Validation Loss: 0.3577

========== Hyperparameter Combination [7/8] ==========
Hidden Size: 128 | Dropout: 0.3 | Learning Rate: 0.001
Best Validation Loss: 0.3547

========== Hyperparameter Combination [8/8] ==========
Hidden Size: 128 | Dropout: 0.3 | Learning Rate: 0.0005
Best Validation Loss: 0.3695

=========FINISHING 

,hidden_size,dropout,learning_rate,best_val_loss,epochs,val_roc_auc,val_pr_auc
4,128,0.2,0.0010,0.350557,92.0,0.838084,0.917427
6,128,0.3,0.0010,0.354728,115.0,0.837521,0.917007
7,128,0.3,0.0005,0.369521,103.0,0.828252,0.912364
5,128,0.2,0.0005,0.357685,99.0,0.827718,0.911586
0,64,0.2,0.0010,0.411297,127.0,0.826066,0.912065
2,64,0.3,0.0010,0.431833,86.0,0.824396,0.911748
3,64,0.3,0.0005,0.434878,134.0,0.822530,0.910758
1,64,0.2,0.0005,0.428815,129.0,0.821926,0.909336


Now we can select the model (and hyperparameters) we want to use going forward on the test set. As shown in the code below, we chose to pick the model that **achieved the highest validation ROC-AUC score**.

In [19]:
# Locate the row index of the optimal model configuration based on the highest ROC-AUC score
best_model_row_idx = param_combos_df['val_roc_auc'].idxmax()

# Extract and store the trained model object from the optimal model row index
best_model = param_combos_df.at[best_model_row_idx, 'model_object']

# Extract and store the optimal hyperparameter values (hidden size, dropout, learning rate)
best_params = param_combos_df.loc[best_model_row_idx, ['hidden_size', 'dropout', 'learning_rate']].to_dict()

# Combine the fixed hyperparameters (window size and batch size) with the extracted hyperparameters in a DataFrame
best_params = pd.DataFrame({'window_size': WINDOW_SIZE, 'batch_size': BATCH_SIZE, **best_params}, index=['value'])

# Display the optimal model hyperparameters and model
print("Chosen Model Parameters:")
display(best_params)
print("\nChosen Model:")
display(best_model)

Chosen Model Parameters:


,window_size,batch_size,hidden_size,dropout,learning_rate
value,10,64,128.0,0.2,0.001



Chosen Model:


OptimizedModule(
  (_orig_mod): LSTMAutoencoder(
    (encoder): LSTMEncoder(
      (lstm1): LSTM(12, 128, batch_first=True)
      (dropout1): Dropout(p=0.2, inplace=False)
      (lstm2): LSTM(128, 64, batch_first=True)
      (dropout2): Dropout(p=0.2, inplace=False)
      (lstm3): LSTM(64, 32, batch_first=True)
    )
    (decoder): LSTMDecoder(
      (lstm1): LSTM(32, 64, batch_first=True)
      (dropout1): Dropout(p=0.2, inplace=False)
      (lstm2): LSTM(64, 128, batch_first=True)
      (dropout2): Dropout(p=0.2, inplace=False)
      (lstm3): LSTM(128, 128, batch_first=True)
      (output_layer): Linear(in_features=128, out_features=12, bias=True)
    )
  )
)

References:
* ApX Machine Learning. (n.d.). *Tuning hyperparameters for optimal performance*. https://apxml.com/courses/applied-autoencoders-feature-extraction/chapter-7-applying-autoencoder-features-practical-guidance/tuning-hyperparameters-optimal-performance



### 5.5. Plot Training/Validation Loss Curves for the Chosen Model

In [20]:
# Extract and store the historical training/validation loss lists from the optimal model row index
best_train_losses = param_combos_df.at[best_model_row_idx, 'train_losses']
best_val_losses = param_combos_df.at[best_model_row_idx, 'val_losses']

# Create a DataFrame containing the optimal model's training/validation losses and epochs
loss_curve_df = pd.DataFrame({
  'Epoch': range(1, len(best_train_losses) + 1),
  'Training Loss': best_train_losses,
  'Validation Loss': best_val_losses
})

# Create a Plotly line chart to depict the optimal model's train/validation losses over epochs
loss_curve_fig = px.line(
  loss_curve_df,
  x='Epoch',
  y=['Training Loss', 'Validation Loss'],
  markers=True,
  title="Optimal Model's Residual Training & Validation Loss Over Epochs",
  labels={'value': 'Residual Loss (MAE)', 'variable': 'Loss Type'}
)

# Update the line chart's plot size and display it
loss_curve_fig.update_layout(height=500, width=750)
loss_curve_fig.show()

### 5.6. Obtain the Validation Set Anomaly Scores and ROC-AUC/PR-AUC Metrics of the Chosen Model

Now that we have the optimized model, we evaluate validation performance using the full dataset of both healthy and degreaded/unhealthy engine states.

In [21]:
# Evaluate the determined optimal model on the validation set
val_roc_auc, val_pr_auc, val_anomaly_scores, val_anomaly_labels = evaluate_lstm_autoencoder(best_model, val_full_loader, val_full_data)

# Create a comprehensive DataFrame to store and inspect validation results and other info
val_results_df = pd.DataFrame({
  'Engine/Unit': val_full_data['unit_ids'],
  'Fc': val_full_data['flight_conditions'],
  'Sliding Window Index Range': val_full_data['window_ranges'],
  'Sliding Window End Index': val_full_data['window_end_steps'],
  'Residual Reconstruction Error (Cumulative MAE)': val_anomaly_scores,
  'Health Status (Numeric)': val_anomaly_labels,
  'Health Status': np.where(val_anomaly_labels == 0, 'Normal', 'Anomaly')
})

# Create a comprehensive validation evaluation performance summary DataFrame
val_eval_scores_df = create_segmented_metric_summary(val_results_df, val_anomaly_labels, val_anomaly_scores, val_roc_auc, val_pr_auc)

# Display the first five rows of the validation results DataFrame
print("\nOptimal Model Validation Results (First 5 Rows):")
display(val_results_df.head())

# Display the comprehensive validation evaluation DataFrame
print("\nOptimal Model Validation Evaluation Metrics:")
display(val_eval_scores_df)


Optimal Model Validation Results (First 5 Rows):


,Engine/Unit,Fc,Sliding Window Index Range,Sliding Window End Index,Residual Reconstruction Error (Cumulative MAE),Health Status (Numeric),Health Status
0,4,2,1-10,10,3.394568,0.0,Normal
1,4,2,2-11,11,3.224643,0.0,Normal
2,4,2,3-12,12,3.054718,0.0,Normal
3,4,2,4-13,13,2.996657,0.0,Normal
4,4,2,5-14,14,2.983865,0.0,Normal



Optimal Model Validation Evaluation Metrics:


,ROC-AUC,PR-AUC
Overall,0.838100,0.917400
Fc = 1,0.904958,0.942574
Fc = 2,0.846947,0.928579
Fc = 3,0.752939,0.894053


### 5.7. Determine the Recall-Focused Alarm Threshold

As mentioned in our work for the Isolation Forest model, because missing a degrading engine is significantly considered costlier than conducting extra maintenance checks, this project is **recall-focused**: we want to catch instances of degrading/unhealthy engines while accepting some false alarms. As such, we need to define a locked alarm decision threshold to evaluate test set performance, switching continuous reconstruction error scores over the sliding windows into discrete binary alarm decisions/classifications.

In [22]:
def extract_recall_threshold(anomaly_labels, anomaly_scores, target_recall):
  """Finds the highest threshold whose recall on (anomaly_labels, anomaly_scores) is >= target_recall"""
  fpr, tpr, thresholds = roc_curve(anomaly_labels, anomaly_scores)
  return thresholds[np.argmax(tpr >= target_recall)]

# Set a 90% recall target
TARGET_RECALL = 0.90

# Extract the alarm threshold targeting 90% recall from the validation set
alarm_threshold = extract_recall_threshold(
  val_results_df['Health Status (Numeric)'].values,
  val_results_df['Residual Reconstruction Error (Cumulative MAE)'].values,
  target_recall=TARGET_RECALL
)

print(f"Locked Alarm Threshold (Chosen on Validation Set, Target Recall = {TARGET_RECALL:.0%}): {alarm_threshold:.4f}")

Locked Alarm Threshold (Chosen on Validation Set, Target Recall = 90%): 3.3919


## 6.0. Evaluating the Chosen Model on the Test Set

### 6.1. Obtain the Test Set Anomaly Scores and ROC-AUC/PR-AUC Metrics Using the Chosen Model

In [23]:
# Create the residual sliding window tensors for the test set
test_data = create_residual_sliding_windows(test, RESIDUAL_COLUMNS, window_size=WINDOW_SIZE)

# Wrap the test tensors in a PyTorch dataset and create the test DataLoader
test_dataset = TensorDataset(test_data['tensor_residuals'])
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [24]:
# Evaluate the determined optimal model on the unseen test set
test_roc_auc, test_pr_auc, test_anomaly_scores, test_anomaly_labels = evaluate_lstm_autoencoder(best_model, test_loader, test_data)

# Create a comprehensive DataFrame to store and inspect test results and other info
test_results_df = pd.DataFrame({
  'Engine/Unit': test_data['unit_ids'],
  'Fc': test_data['flight_conditions'],
  'Sliding Window Index Range': test_data['window_ranges'],
  'Sliding Window End Index': test_data['window_end_steps'],
  'Residual Reconstruction Error (Cumulative MAE)': test_anomaly_scores,
  'Health Status (Numeric)': test_anomaly_labels,
  'Health Status': np.where(test_anomaly_labels == 0, 'Normal', 'Anomaly')
})

# Create a comprehensive test evaluation performance summary DataFrame
test_eval_scores_df = create_segmented_metric_summary(test_results_df, test_anomaly_labels, test_anomaly_scores, test_roc_auc, test_pr_auc)

# Display the first five rows of the test results DataFrame
print("\nOptimal Model Test Results (First 5 Rows):")
display(test_results_df.head())

# Display the comprehensive test evaluation DataFrame
print("\nOptimal Model Test Evaluation Metrics:")
display(test_eval_scores_df)


Optimal Model Test Results (First 5 Rows):


,Engine/Unit,Fc,Sliding Window Index Range,Sliding Window End Index,Residual Reconstruction Error (Cumulative MAE),Health Status (Numeric),Health Status
0,1,1,1-10,10,5.853924,0.0,Normal
1,1,1,2-11,11,4.633565,0.0,Normal
2,1,1,3-12,12,3.594968,0.0,Normal
3,1,1,4-13,13,3.594968,0.0,Normal
4,1,1,5-14,14,3.594968,0.0,Normal



Optimal Model Test Evaluation Metrics:


,ROC-AUC,PR-AUC
Overall,0.797800,0.900300
Fc = 1,0.885965,0.931270
Fc = 2,0.804795,0.904331
Fc = 3,0.716356,0.881750


### 6.2. Evaluate Alarm Performance on the Test Set

Now that we have the model test set results using the optimal model hyperparameters and an alarm decision threshold (extracted from the validation set), we can evaluate the classification performance at each sliding window end-index.

We calculate the same operational metrics as with the Isolation Forest model. These metrics include **Recall**, **False Positive Rate (FPR)**, **Precision**, and the **$F_2$-Score**. Confusion matrix counts (**false negatives**, **false positives**, **true negatives**, and **true positives**) are computed as well. We display the the counts and metrics for analysis afterwards.

In [25]:
def evaluate_alarm_performance(anomaly_labels, anomaly_scores, alarm_threshold):
  """
  Evaluates alarm performance by applying a locked threshold to dataset anomaly scores and
  calculating confusion matrix counts and classification metrics at the sliding window level
  """
  # Convert the anomaly scores into binary alarm flags based off of threshold (1 = Alarm, 0 = Normal)
  alarms_triggered = (anomaly_scores >= alarm_threshold).astype(int)

  # Package alarm confusion matrix counts into a dictionaey
  confusion_matrix_counts = {
    'True Positives (Degradation Caught)': ((alarms_triggered == 1) & (anomaly_labels == 1)).sum(),
    'False Negatives (Degradation Missed)': ((alarms_triggered == 0) & (anomaly_labels == 1)).sum(),
    'False Positives (False Alarms)': ((alarms_triggered == 1) & (anomaly_labels == 0)).sum(),
    'True Negatives (Healthy Correctly Ignored)': ((alarms_triggered == 0) & (anomaly_labels == 0)).sum()
  }

  # Package alarm recall, FPR, precision, and F2-score metrics into a dictionary
  classification_metrics = {
    'Recall': recall_score(anomaly_labels, alarms_triggered, zero_division=0),
    'FPR': (((alarms_triggered == 1) & (anomaly_labels == 0)).sum() / (anomaly_labels == 0).sum()) if (anomaly_labels == 0).sum() > 0 else 0,
    'Precision': precision_score(anomaly_labels, alarms_triggered, zero_division=0),
    'F2-Score': fbeta_score(anomaly_labels, alarms_triggered, beta=2, zero_division=0),
  }
  return confusion_matrix_counts, classification_metrics

# Extract and store dictionaries of alarm confusion matrix counts and classification metrics for the test set
test_alarm_counts, test_alarm_metrics = evaluate_alarm_performance(
  test_results_df['Health Status (Numeric)'].values,
  test_results_df['Residual Reconstruction Error (Cumulative MAE)'].values,
  alarm_threshold
)

# Convert dictionaries into structured pandas DataFrames and display them
test_alarm_counts_df = pd.DataFrame(list(test_alarm_counts.items()), columns=['Confusion Matrix Category', 'Count'])
test_alarm_metrics_df = pd.DataFrame(list(test_alarm_metrics.items()), columns=['Classification Metric', 'Score'])

print("\nTest Set Alarm Confusion Matrix Counts (Per Sliding Window End Index):")
display(test_alarm_counts_df)

print("\nTest Set Alarm Classifiction Metrics (Per Sliding Window End Index):")
display(test_alarm_metrics_df)


Test Set Alarm Confusion Matrix Counts (Per Sliding Window End Index):


,Confusion Matrix Category,Count
0,True Positives (Degradation Caught),7425
1,False Negatives (Degradation Missed),1231
2,False Positives (False Alarms),2575
3,True Negatives (Healthy Correctly Ignored),1999



Test Set Alarm Classifiction Metrics (Per Sliding Window End Index):


,Classification Metric,Score
0,Recall,0.857787
1,FPR,0.562965
2,Precision,0.742500
3,F2-Score,0.831951


### 6.3. Plot the Test Anomaly Score Distributions Across Engine Health Status Categories

In [26]:
# Create a Plotly box plot to depict the distribution and median cumlative anomaly scores across health status categories
box_plot_fig = px.box(
  test_results_df,
  x='Residual Reconstruction Error (Cumulative MAE)',
  y='Health Status',
  color='Health Status',
  points="all",
  title="Anomaly Score Distributions (Box Plot)"
)

# Create a Plotly histogram to depict the overlapping frequency densities of cumulative anomaly scores across health status categories
hist_plot_fig = px.histogram(
  test_results_df,
  x='Residual Reconstruction Error (Cumulative MAE)',
  color='Health Status',
  barmode='overlay',
  opacity=0.6,
  nbins=50,
  title="Anomaly Score Distributions (Histogram)"
)

# Update the sizes of the box plot and histogram visualizations
hist_plot_fig.update_layout(height=500, width=800)
box_plot_fig.update_layout(height=500, width=800)

# Display the anomaly score distribution plots (box plot and histogram)
box_plot_fig.show()
hist_plot_fig.show()

### 6.4. Plot the Test ROC and Precision-Recall Curves with Baselines

In [27]:
# Create 2 subplots for the ROC/PR curves and include AUC scores inside the subplot titles
model_eval_curves = make_subplots(
  rows=1,
  cols=2,
  subplot_titles=(f"ROC-AUC Curve (AUC = {test_roc_auc:.4f})", f"PR-AUC Curve (AUC = {test_pr_auc:.4f})",)
)

# Calculate FPR, TPR, precision, and recall across thresholds for the ROC and PR curves
false_pos_rate, true_pos_rate, _ = roc_curve(test_anomaly_labels, test_anomaly_scores)
precision, recall, _ = precision_recall_curve(test_anomaly_labels, test_anomaly_scores)

# Calculate the positive class ratio to use as the random chance baseline for the PR curve
pr_baseline = test_anomaly_labels.sum() / len(test_anomaly_labels)

# Populate and update the subplot for the ROC curve and add add random chance baseline
model_eval_curves.add_trace(
  go.Scatter(x=false_pos_rate, y=true_pos_rate, mode='lines', name='ROC', fill='tozeroy'),
  row=1,
  col=1
)
model_eval_curves.add_trace(
  go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash'), name='Random Chance (AUC = 0.5)'),
  row=1,
  col=1
)

# Populate and update the subplot for the PR curve and add random chance baseline
model_eval_curves.add_trace(
  go.Scatter(x=recall, y=precision, mode='lines', name='PR', fill='tozeroy'),
  row=1,
  col=2
)
model_eval_curves.add_trace(
  go.Scatter(x=[0, 1], y=[pr_baseline, pr_baseline], mode='lines', line=dict(dash='dash'), name=f'Random Chance (AUC = {pr_baseline:.3f})'),
  row=1,
  col=2
)

# Update the x-axis/y-axis labels and bounds of the subplots
model_eval_curves.update_xaxes(title_text="False Positive Rate", col=1)
model_eval_curves.update_yaxes(title_text="True Positive Rate", col=1)
model_eval_curves.update_xaxes(title_text="Recall", col=2)
model_eval_curves.update_yaxes(title_text="Precision", col=2)

# Update the overall plot's size, title, and axes bounds
model_eval_curves.update_layout(
    title_text="Chosen Model's Anomaly Score Evaluation Curves (Residual Cumulative MAE)",
    xaxis=dict(range=[0, 1.01], autorange=False),
    yaxis=dict(range=[0, 1.01], autorange=False),
    height=500, width=1100
)

# Display the subplots
model_eval_curves.show()

### 6.5. Plot the Anomaly Score Trajectories Over Flight Cycle Sliding Windows

In [28]:
# Plot score trajectory lines over flight cycle sliding windows for ALL test engines
score_trag_fig = px.line(
  test_results_df,
  x='Sliding Window End Index',
  y='Residual Reconstruction Error (Cumulative MAE)',
  color='Health Status',
  facet_col='Engine/Unit',
  facet_col_wrap=3,
  markers=True,
  title="Anomaly Score Trajectories Across All Test Engines"
)

# Update the anomaly score trajectory plot's size and y-axis labels
score_trag_fig.update_yaxes(title_text="", col=2)
score_trag_fig.update_yaxes(title_text="", col=3)
score_trag_fig.update_layout(height=500, width=1200)

# Display the anomaly score trajectory plot
score_trag_fig.show()

The anomaly score trajectories across the test engines (Unit 1, Unit 11, and Unit 12) show the progression of cumulative reconstruction error (MAE) over the 10-step sliding windows. As each of the test engines approach their end-of-life and/or become degraded/unhealthy, the anomaly scores display a sharp, non-linear exponential trajectory upwards. This behavior is expected and showcases our LSTM autoencoder's ability to detect cumulative engine degradation over time.